In [4]:
import kagglehub
import polars as pl
import pandas as pd
import time 

#download dataset
path = kagglehub.dataset_download("jeandedieunyandwi/lending-club-dataset")
print(path)  # see where it saved


FILEPATH = "/Users/Sophie/.cache/kagglehub/datasets/jeandedieunyandwi/lending-club-dataset/versions/1/lending_club_loan_two.csv"

# load dataset using polars and pandas
df_pl = pl.read_csv(FILEPATH, infer_schema_length=10000)
df_pd = pd.read_csv(FILEPATH)

print(df_pl.shape)
print(df_pd.shape)  # should be identical

/Users/Sophie/.cache/kagglehub/datasets/jeandedieunyandwi/lending-club-dataset/versions/1
(396030, 27)
(396030, 27)


In [ ]:
path = kagglehub.dataset_download("jeandedieunyandwi/lending-club-dataset")
print(path)
print(os.listdir(path))  # shows exactly what files are in the folder

100%|██████████| 27.6M/27.6M [00:03<00:00, 7.60MB/s]

Extracting files...


/Users/Sophie/.cache/kagglehub/datasets/jeandedieunyandwi/lending-club-dataset/versions/1
['lending_club_loan_two.csv']


In [5]:
t0 = time.time()
_ = pd.read_csv(FILEPATH)
print(f"pandas read: {time.time()-t0:.2f}s")

t0 = time.time()
_ = pl.read_csv(FILEPATH, infer_schema_length=10000)
print(f"polars read: {time.time()-t0:.2f}s")

pandas read: 0.59s
polars read: 0.06s


In [9]:
# pandas
t0 = time.time()
result_pd = df_pd.groupby('grade')['loan_amnt'].mean()
pd_time = time.time() - t0
print(f"pandas groupby: {pd_time:.4f}s")

# polars
t0 = time.time()
result_pl = df_pl.group_by('grade').agg(pl.col('loan_amnt').mean())
pl_time = time.time() - t0
print(f"polars groupby: {pl_time:.4f}s")

print(f"speedup: {pd_time/pl_time:.3f}x")

pandas groupby: 0.0088s
polars groupby: 0.0032s
speedup: 2.781x


In [8]:
# pandas
t0 = time.time()
df_pd['loan_to_income'] = df_pd['loan_amnt'] / df_pd['annual_inc']
df_pd['installment_to_income'] = df_pd['installment'] / (df_pd['annual_inc'] / 12)
df_pd['term_clean'] = df_pd['term'].str.replace('months','').astype(float)
pd_time = time.time() - t0
print(f"pandas feature engineering: {pd_time:.4f}s")

# polars
t0 = time.time()
df_pl = df_pl.with_columns([
    (pl.col('loan_amnt') / pl.col('annual_inc')).alias('loan_to_income'),
    (pl.col('installment') / (pl.col('annual_inc') / 12)).alias('installment_to_income'),
    pl.col('term').str.extract(r'(\d+)').cast(pl.Float64).alias('term_clean')
])
pl_time = time.time() - t0
print(f"polars feature engineering: {pl_time:.4f}s")

print(f"speedup: {pd_time/pl_time:.1f}x")

pandas feature engineering: 0.0339s
polars feature engineering: 0.0235s
speedup: 1.4x
